Approach 4: retrieval and label conditioned generation
Owner: Yousef

In [ ]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((p.resolve() for p in CANDIDATES if (p / "src" / "config.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the cxr_second_reader project folder in VS Code/Jupyter.")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Local project folder:", REPO_ROOT)


In [ ]:
from src.data import load_table, CXRDataset
train_df = load_table("train")
val_df = load_table("val")
test_df = load_table("test")
print(f"Studies/rows — train: {len(train_df)}, validation: {len(val_df)}, test: {len(test_df)}")
print("Columns:", train_df.columns.tolist())
print("Target example:", train_df.iloc[0]["target_text"][:300])


In [ ]:
# Text-only retrieval control: query with indication (available input), not the target report.
# Fit on training reports only; this is not image retrieval or a clinical system.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from src.evaluate import evaluate, log_result

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=50000)
train_text = train_df["target_text"].fillna("").astype(str).tolist()
train_queries = train_df["indication"].fillna("").astype(str).tolist()
test_text = test_df["target_text"].fillna("").astype(str).tolist()
test_queries = test_df["indication"].fillna("").astype(str).tolist()
fallback = "no indication provided"
train_queries = [q.strip() or fallback for q in train_queries]
test_queries = [q.strip() or fallback for q in test_queries]
train_matrix = vectorizer.fit_transform(train_text)
test_query_matrix = vectorizer.transform(test_queries)
similarity = cosine_similarity(test_query_matrix, train_matrix)
retrieved = [train_text[i] for i in similarity.argmax(axis=1)]
print("Test studies with an indication-based training-report retrieval:", len(retrieved))
print("Mean best-match cosine similarity:", round(float(similarity.max(axis=1).mean()), 4))


In [ ]:
metrics = evaluate(retrieved, test_text)
log_result("05_text_tfidf_retrieval_control", metrics)
print(metrics)
